In [2]:
import pdfplumber
import pandas as pd
import re
from datetime import datetime

# Ganti dengan path file PDF Anda
pdf_path = 'Bank File/Bank.pdf' 

all_text = ''
all_tables = []
header = None

# Buka file PDF
with pdfplumber.open(pdf_path) as pdf:
    # Loop untuk setiap halaman di dalam PDF
    for page_num, page in enumerate(pdf.pages):
        # Ekstrak semua tabel dari halaman tersebut
        tables_on_page = page.extract_tables()
        
        for table in tables_on_page:
            if table:  # Pastikan tabel tidak kosong
                if page_num == 0:  # Halaman pertama
                    # Ambil header dari halaman pertama
                    if header is None:
                        header = table[0]
                    # Ambil semua data setelah header
                    all_tables.extend(table[1:])
                else:  # Halaman kedua dan seterusnya
                    # Periksa apakah baris pertama adalah header yang sama
                    if len(table) > 0:
                        first_row = table[0]
                        # Bandingkan dengan header yang sudah ada
                        if header and first_row == header:
                            # Skip header dan ambil data mulai dari baris kedua
                            if len(table) > 1:
                                all_tables.extend(table[1:])
                        else:
                            # Jika baris pertama bukan header yang sama, anggap sebagai data
                            all_tables.extend(table)

# Buat DataFrame dengan header yang sudah diambil dari halaman pertama
if header and all_tables:
    df_bank = pd.DataFrame(all_tables, columns=header)
    
    # Cari kolom yang kemungkinan berisi narasi
    narasi_keywords = ['Narasi']
    narasi_columns = [col for col in df_bank.columns if any(keyword in str(col).lower() for keyword in narasi_keywords)]
    
    if narasi_columns:
        for col in narasi_columns:
            # Hapus semua \n dan \r dari kolom narasi
            df_bank[col] = df_bank[col].astype(str).str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)
            # Bersihkan juga spasi berlebih
            df_bank[col] = df_bank[col].str.replace(r'\s+', '', regex=True).str.strip()
    else:
        # Membersihkan semua kolom string dari karakter newline
        for col in df_bank.columns:
            if df_bank[col].dtype == 'object':  # Kolom string
                df_bank[col] = df_bank[col].astype(str).str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)
                df_bank[col] = df_bank[col].str.replace(r'\s+', '', regex=True).str.strip()
    
    # PEMBERSIHAN KOLOM CREDIT TRANSACTION DAN BALANCE
    
    # Fungsi untuk membersihkan format angka (hilangkan 2 angka di belakang koma dan koma)
    def clean_numeric_column(value):
       
        if pd.isna(value) or value == '' or value == 'nan':
            return value
        
        value_str = str(value).strip()
        
        # Jika ada titik desimal, hilangkan 2 angka terakhir (desimal)
        if '.' in value_str:
            # Pisahkan bagian sebelum dan sesudah titik
            parts = value_str.split('.')
            if len(parts) == 2 and len(parts[1]) >= 2:
                # Hilangkan 2 angka terakhir (desimal)
                value_str = parts[0]
        
        # Hilangkan semua koma
        value_str = value_str.replace(',', '')
        
        return value_str
    
    # Daftar kolom yang perlu dibersihkan
    columns_to_clean = ['Credit Transaction', 'Balance']
    
    # Cek dan bersihkan kolom yang ada
    for col_name in columns_to_clean:
        # Cari kolom yang mengandung nama tersebut (case insensitive)
        matching_cols = [col for col in df_bank.columns if col_name.lower() in str(col).lower()]
        
        if matching_cols:
            for col in matching_cols:
                # Terapkan pembersihan
                df_bank[col] = df_bank[col].apply(clean_numeric_column)
    
    # PEMBERSIHAN DAN FORMAT KOLOM TANGGAL
    
    # Fungsi untuk memformat tanggal menjadi dd/mm/yyyy
    def format_date_column(value):
        """
        Memformat kolom tanggal menjadi format dd/mm/yyyy
        Menangani berbagai format input tanggal termasuk format DDMmmYYYY
        """
        if pd.isna(value) or value == '' or value == 'nan':
            return value
        
        value_str = str(value).strip()
        
        # Jika sudah dalam format yang benar, kembalikan
        if re.match(r'^\d{2}/\d{2}/\d{4}$', value_str):
            return value_str
        
        # Handle format DDMmmYYYY (seperti 03Jun2025)
        month_mapping = {
            'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
            'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12',
            'JAN': '01', 'FEB': '02', 'MAR': '03', 'APR': '04', 'MAY': '05', 'JUN': '06',
            'JUL': '07', 'AUG': '08', 'SEP': '09', 'OCT': '10', 'NOV': '11', 'DEC': '12'
        }
        
        # Pattern untuk DDMmmYYYY (03Jun2025)
        ddmmmyyyy_pattern = r'(\d{2})([A-Za-z]{3})(\d{4})'
        match = re.match(ddmmmyyyy_pattern, value_str)
        if match:
            day, month_str, year = match.groups()
            month_num = month_mapping.get(month_str)
            if month_num:
                return f"{day}/{month_num}/{year}"
        
        # Coba parse berbagai format tanggal
        date_formats = [
            '%d/%m/%Y',    # 13/06/2025
            '%d-%m-%Y',    # 13-06-2025  
            '%Y-%m-%d',    # 2025-06-13
            '%d/%m/%y',    # 13/06/25
            '%d-%m-%y',    # 13-06-25
            '%d %m %Y',    # 13 06 2025
            '%d.%m.%Y',    # 13.06.2025
            '%d%b%Y',      # 13Jun2025
            '%d-%b-%Y',    # 13-Jun-2025
        ]
        
        for fmt in date_formats:
            try:
                # Parse tanggal dengan format yang dicoba
                parsed_date = datetime.strptime(value_str, fmt)
                # Return dalam format dd/mm/yyyy
                return parsed_date.strftime('%d/%m/%Y')
            except ValueError:
                continue
        
        # Jika tidak bisa di-parse, coba ekstrak angka dan format ulang
        # Pattern untuk mencari tanggal dalam berbagai format
        date_patterns = [
            r'(\d{1,2})[/-](\d{1,2})[/-](\d{4})',  # dd/mm/yyyy atau dd-mm-yyyy
            r'(\d{4})[/-](\d{1,2})[/-](\d{1,2})',  # yyyy/mm/dd atau yyyy-mm-dd
            r'(\d{1,2})[/-](\d{1,2})[/-](\d{2})',  # dd/mm/yy atau dd-mm-yy
        ]
        
        for pattern in date_patterns:
            match = re.search(pattern, value_str)
            if match:
                groups = match.groups()
                
                if len(groups[2]) == 4:  # Format dengan tahun 4 digit
                    if len(groups[0]) == 4:  # yyyy/mm/dd
                        year, month, day = groups[0], groups[1], groups[2]
                    else:  # dd/mm/yyyy
                        day, month, year = groups[0], groups[1], groups[2]
                else:  # Format dengan tahun 2 digit
                    day, month, year = groups[0], groups[1], groups[2]
                    # Konversi tahun 2 digit ke 4 digit
                    year_int = int(year)
                    if year_int >= 20 and year_int <= 30:
                        year = '20' + year
                    else:
                        year = '20' + year  # Default ke 20xx
                
                # Pastikan format dengan leading zero
                day = day.zfill(2)
                month = month.zfill(2)
                
                # Validasi tanggal
                try:
                    datetime.strptime(f"{day}/{month}/{year}", '%d/%m/%Y')
                    return f"{day}/{month}/{year}"
                except ValueError:
                    continue
        
        # Jika semua gagal, kembalikan nilai asli
        return value_str
    
    # Daftar kolom tanggal yang perlu diformat
    date_columns = ['Posting Date', 'Effective Date']
    
    # Format kolom tanggal
    for col_name in date_columns:
        # Cari kolom yang mengandung nama tersebut (case insensitive)
        matching_cols = [col for col in df_bank.columns if col_name.lower() in str(col).lower()]
        
        if matching_cols:
            for col in matching_cols:
                print(f"📅 Memformat kolom tanggal: {col}")
                # Terapkan format tanggal
                df_bank[col] = df_bank[col].apply(format_date_column)
    
    # Set pandas options untuk menampilkan teks lengkap
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_columns', None)
    
    # Tampilkan beberapa baris pertama
    display(df_bank.head(10))
else:
    pass

📅 Memformat kolom tanggal: Posting Date
📅 Memformat kolom tanggal: Effective Date


,No,Posting Date,Effective Date,Narasi,Debit Transaction,Credit Transaction,Balance
0,1,01/07/2025,01/07/2025,0023-800476-360/L704-0901SETORTUNAI/MARTINAVENTYDEYRUSUN03020308/BIII8/MARTINA/JUNI25,,430000,430000
1,2,01/07/2025,01/07/2025,0023-800474-360/L778-0535SETORTUNAI/SUSANROSADIQQMUHAMMA01010205/SUSANROSADI/AII5/JUNI2025,,402900,832900
2,3,01/07/2025,01/07/2025,0175-800459-360/K565-0175SETORTUNAI/IIMROSMINIRSNCIBEUREUM/02040203/JUHANA/DII3/MEI2025,,456450,1289350
3,4,01/07/2025,01/07/2025,0262-800459-360/J580-0262SETORTUNAI/RINA,,453900,1743250
4,5,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/DES24RUSUN/02040104/DUDE/DI4/DES24,,462500,2205750
5,6,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/JAN25RUSUN/02040104/DUDE/DI4/JAN25,,462500,2668250
6,7,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/FEB25RUSUN/02040104/DUDE/DI4/FEB25,,462500,3130750
7,8,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/JUN25RUSUN/02040104/DUDE/DI4/JUN25,,462500,3593250
8,9,01/07/2025,01/07/2025,0262-800459-360/J580-0262SETORTUNAI/RIFKYIININDRIANI/03030518/JUNI2025IININDRIANI/03030518/JUNI2025,,355000,3948250
9,10,01/07/2025,01/07/2025,0023-800474-360/L778-0535SETORTUNAI/YUSEP,,357000,4305250


In [3]:
# REVISI REGEX KHUSUS UNTUK DATA SETORTUNAI
print("🔄 === REVISI REGEX UNTUK DATA SETORTUNAI ===")

# Pisahkan data berdasarkan kata SETORTUNAI
df_bank['Has_SETORTUNAI'] = df_bank['Narasi'].str.upper().str.contains('SETORTUNAI', na=False)

# Data dengan SETORTUNAI (akan diproses untuk ekstraksi)
df_setortunai = df_bank[df_bank['Has_SETORTUNAI'] == True].copy()

# Data tanpa SETORTUNAI (akan disimpan sebagai non-rusun)
df_non_rusun = df_bank[df_bank['Has_SETORTUNAI'] == False].copy()

print(f"Total data: {len(df_bank)}")
print(f"Data dengan SETORTUNAI: {len(df_setortunai)} baris")
print(f"Data NON-RUSUN (tanpa SETORTUNAI): {len(df_non_rusun)} baris")

# Fungsi ekstraksi yang diperbaiki berdasarkan analisis contoh narasi
def extract_setortunai_only(narasi_text):
    """
    Fungsi ekstraksi yang diperbaiki untuk menangani pattern NOV25, DES24, dll
    dan tidak memberikan default bulan/tahun jika tidak ada indikasi yang jelas
    """
    if pd.isna(narasi_text) or narasi_text == '' or narasi_text == 'nan':
        return '', '', ''
    
    narasi_str = str(narasi_text).upper().strip()
    
    # Hanya proses jika mengandung SETORTUNAI
    if 'SETORTUNAI' not in narasi_str:
        return '', '', ''
    
    # === KODE 8 DIGIT ===
    kode_8_digit = ''
    
    # Cari semua 8 digit dalam narasi dengan pattern yang lebih spesifik
    # 2 angka awal harus 01, 02, atau 03
    specific_8_digits = re.findall(r'\b(0[123]\d{6})\b', narasi_str)
    
    if specific_8_digits:
        # Ambil yang pertama ditemukan yang sesuai kriteria
        kode_8_digit = specific_8_digits[0]
    else:
        # Fallback: cari semua 8 digit dan filter yang dimulai 01, 02, 03
        all_8_digits = re.findall(r'\d{8}', narasi_str)
        
        for code in all_8_digits:
            # Filter kode yang valid: dimulai dengan 01, 02, atau 03 dan bukan tahun
            if (code.startswith(('01', '02', '03')) and 
                not code.startswith(('2024', '2025', '2026'))):
                kode_8_digit = code
                break
    
    # === ANALISIS 40 HURUF TERAKHIR ===
    last_40_chars = narasi_str[-40:] if len(narasi_str) >= 40 else narasi_str
    
    # === BULAN - PERBAIKAN DENGAN PRIORITAS YANG TEPAT ===
    bulan = ''
    
    # 1. PRIORITAS TERTINGGI: Cari nama bulan lengkap + tahun 4 digit di 40 huruf terakhir
    full_month_year_4digit = r'(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)(20\d{2})'
    matches = re.findall(full_month_year_4digit, last_40_chars)
    if matches:
        month_name = matches[-1][0]  # Ambil nama bulan
        month_dict = {
            'JANUARI': 'Januari', 'FEBRUARI': 'Februari', 'MARET': 'Maret', 'APRIL': 'April',
            'MEI': 'Mei', 'JUNI': 'Juni', 'JULI': 'Juli', 'AGUSTUS': 'Agustus',
            'SEPTEMBER': 'September', 'OKTOBER': 'Oktober', 'NOVEMBER': 'November', 'DESEMBER': 'Desember'
        }
        bulan = month_dict.get(month_name, '')
    
    # 2. PRIORITAS KEDUA: Cari pattern bulan singkat 3 huruf + tahun 4 digit (FEB2025, JAN2025, DES2024)
    if not bulan:
        short_month_year_4digit = r'(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGS|AGST|SEP|SEPT|OKT|NOV|DES)(20\d{2})'
        matches = re.findall(short_month_year_4digit, last_40_chars)
        if matches:
            month_abbr = matches[-1][0]  # Ambil yang terakhir ditemukan
            short_month_dict = {
                'JAN': 'Januari', 'FEB': 'Februari', 'MAR': 'Maret', 'APR': 'April',
                'MEI': 'Mei', 'JUN': 'Juni', 'JUL': 'Juli', 'AGS': 'Agustus', 'AGST': 'Agustus',
                'SEP': 'September', 'SEPT': 'September', 'OKT': 'Oktober', 'NOV': 'November', 'DES': 'Desember'
            }
            bulan = short_month_dict.get(month_abbr, '')
    
    # 3. Jika belum ada, cari nama bulan lengkap di seluruh narasi
    if not bulan:
        full_month_names = ['JANUARI', 'FEBRUARI', 'MARET', 'APRIL', 'MEI', 'JUNI', 
                           'JULI', 'AGUSTUS', 'SEPTEMBER', 'OKTOBER', 'NOVEMBER', 'DESEMBER']
        
        for month_name in full_month_names:
            if month_name in narasi_str:
                month_dict = {
                    'JANUARI': 'Januari', 'FEBRUARI': 'Februari', 'MARET': 'Maret', 'APRIL': 'April',
                    'MEI': 'Mei', 'JUNI': 'Juni', 'JULI': 'Juli', 'AGUSTUS': 'Agustus',
                    'SEPTEMBER': 'September', 'OKTOBER': 'Oktober', 'NOVEMBER': 'November', 'DESEMBER': 'Desember'
                }
                bulan = month_dict[month_name]
                break
    
    # 4. Cari pattern bulan singkat + 2 digit tahun (NOV25, DES24, dll)
    if not bulan:
        month_year_short = {
            'JAN': 'Januari', 'FEB': 'Februari', 'MAR': 'Maret', 'APR': 'April',
            'MEI': 'Mei', 'JUN': 'Juni', 'JUL': 'Juli', 'AGS': 'Agustus', 'AGST': 'Agustus',
            'SEP': 'September', 'SEPT': 'September', 'OKT': 'Oktober', 'NOV': 'November', 'DES': 'Desember'
        }
        
        # Pattern: 3 huruf bulan + 2 digit tahun
        month_pattern = r'(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGS|AGST|SEP|SEPT|OKT|NOV|DES)(\d{2})'
        matches = re.findall(month_pattern, last_40_chars)
        
        if matches:
            month_abbr = matches[-1][0]  # Ambil yang terakhir ditemukan
            bulan = month_year_short.get(month_abbr, '')
    
    # 5. Cari pattern BLN + singkatan (existing)
    if not bulan:
        bln_abbreviations = {
            'BLNJAN': 'Januari', 'BLNFEB': 'Februari', 'BLNMAR': 'Maret', 'BLNAPR': 'April',
            'BLNMEI': 'Mei', 'BLNJUN': 'Juni', 'BLNJUL': 'Juli', 'BLNAGST': 'Agustus',
            'BLNSEPT': 'September', 'BLNOKT': 'Oktober', 'BLNNOV': 'November', 'BLNDES': 'Desember'
        }
        
        for bln_code, month_name in bln_abbreviations.items():
            if bln_code in last_40_chars:
                bulan = month_name
                break
    
    # 6. Jika masih belum ada, cari angka bulan di 40 huruf terakhir (tetapi lebih selektif)
    if not bulan:
        # Pattern untuk angka bulan dalam konteks yang jelas
        month_patterns = [
            r'/(\d{1,2})/',        # /MM/ atau /M/
            r'/(\d{1,2})\b',       # /MM atau /M (di akhir)
        ]
        
        for pattern in month_patterns:
            matches = re.findall(pattern, last_40_chars)
            for match in matches:
                if match.isdigit() and 1 <= int(match) <= 12:
                    month_names = ['', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni',
                                 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']
                    bulan = month_names[int(match)]
                    break
            if bulan:
                break
    
    # === TAHUN - PERBAIKAN DENGAN PRIORITAS YANG TEPAT ===
    tahun = ''
    
    # 1. PRIORITAS TERTINGGI: Cari format bulan lengkap + tahun 4 digit DULU (MEI2024, DES2024)
    full_month_year_4digit = r'(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)(20\d{2})'
    matches = re.findall(full_month_year_4digit, last_40_chars)
    if matches:
        year_part = matches[-1][-1]  # Ambil bagian tahun (20XX)
        if year_part.isdigit() and len(year_part) == 4:
            tahun = year_part
    
    # 2. PRIORITAS KEDUA: Cari pattern bulan singkat 3 huruf + tahun 4 digit (FEB2025, JAN2025, DES2024)
    if not tahun:
        short_month_year_4digit = r'(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGS|AGST|SEP|SEPT|OKT|NOV|DES)(20\d{2})'
        matches = re.findall(short_month_year_4digit, last_40_chars)
        if matches:
            year_part = matches[-1][-1]  # Ambil bagian tahun (20XX)
            if year_part.isdigit() and len(year_part) == 4:
                tahun = year_part
    
    # 3. Jika belum ada, cari pattern bulan singkat + 2 digit tahun (NOV25, DES24)
    if not tahun:
        month_pattern = r'(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGS|AGST|SEP|SEPT|OKT|NOV|DES)(\d{2})'
        matches = re.findall(month_pattern, last_40_chars)
        
        if matches:
            year_part = matches[-1][1]  # Ambil bagian tahun
            if year_part.isdigit() and len(year_part) == 2:
                # Konversi 24->2024, 25->2025, dll
                if int(year_part) >= 20 and int(year_part) <= 30:  # Range masuk akal
                    tahun = '20' + year_part
    
    # 4. Cari format bulan lengkap + tahun 2 digit (jika ada)
    if not tahun:
        full_month_year_2digit = r'(JANUARI|FEBRUARI|MARET|APRIL|MEI|JUNI|JULI|AGUSTUS|SEPTEMBER|OKTOBER|NOVEMBER|DESEMBER)(\d{2})'
        matches = re.findall(full_month_year_2digit, last_40_chars)
        if matches:
            year_part = matches[-1][-1]  # Ambil bagian tahun
            if year_part.isdigit() and len(year_part) == 2:
                tahun = '20' + year_part
    
    # 5. Cari pattern BLN + bulan + tahun
    if not tahun:
        bln_pattern = r'BLN(JAN|FEB|MAR|APR|MEI|JUN|JUL|AGST|SEPT|OKT|NOV|DES)(\d{2})'
        matches = re.findall(bln_pattern, last_40_chars)
        if matches:
            year_part = matches[-1][1]
            if year_part.isdigit() and len(year_part) == 2:
                tahun = '20' + year_part
    
    # 6. Cari tahun 4 digit standalone
    if not tahun:
        year_4_digit = re.findall(r'\b(20[0-9]{2})\b', last_40_chars)
        if year_4_digit:
            tahun = year_4_digit[-1]
    
    # 5. PENTING: Tidak memberikan default tahun jika tidak ada indikasi yang jelas
    # Berdasarkan analisis, tidak semua narasi memiliki tahun yang valid
    
    return kode_8_digit, bulan, tahun

# Proses data SETORTUNAI
print(f"\n🔍 === PROSES DATA SETORTUNAI ===")

if len(df_setortunai) > 0:
    # Reset kolom ekstraksi untuk data SETORTUNAI
    extraction_results = df_setortunai['Narasi'].apply(extract_setortunai_only)
    
    df_setortunai['Kode_8_Digit'] = extraction_results.apply(lambda x: x[0])
    df_setortunai['Bulan'] = extraction_results.apply(lambda x: x[1])
    df_setortunai['Tahun'] = extraction_results.apply(lambda x: x[2])
    
    # Hapus kolom helper
    df_setortunai = df_setortunai.drop('Has_SETORTUNAI', axis=1)
    
    # Statistik untuk data SETORTUNAI
    total_setortunai = len(df_setortunai)
    kode_setortunai = (df_setortunai['Kode_8_Digit'] != '').sum()
    bulan_setortunai = (df_setortunai['Bulan'] != '').sum()
    tahun_setortunai = (df_setortunai['Tahun'] != '').sum()
    lengkap_setortunai = ((df_setortunai['Kode_8_Digit'] != '') & 
                          (df_setortunai['Bulan'] != '') & 
                          (df_setortunai['Tahun'] != '')).sum()
    
    print(f"📊 HASIL EKSTRAKSI DATA SETORTUNAI:")
    print(f"Total data SETORTUNAI: {total_setortunai}")
    print(f"🔢 Kode 8 Digit: {kode_setortunai} ({kode_setortunai/total_setortunai*100:.1f}%)")
    print(f"📅 Bulan: {bulan_setortunai} ({bulan_setortunai/total_setortunai*100:.1f}%)")
    print(f"📆 Tahun: {tahun_setortunai} ({tahun_setortunai/total_setortunai*100:.1f}%)")
    print(f"✅ Data Lengkap: {lengkap_setortunai} ({lengkap_setortunai/total_setortunai*100:.1f}%)")
else:
    print("❌ Tidak ada data SETORTUNAI")

# Proses data NON-RUSUN
print(f"\n📋 === PROSES DATA NON-RUSUN ===")

if len(df_non_rusun) > 0:
    # Untuk data non-rusun, kosongkan kolom ekstraksi
    df_non_rusun['Kode_8_Digit'] = ''
    df_non_rusun['Bulan'] = ''
    df_non_rusun['Tahun'] = ''
    
    # Hapus kolom helper
    df_non_rusun = df_non_rusun.drop('Has_SETORTUNAI', axis=1)
    
    print(f"📊 DATA NON-RUSUN:")
    print(f"Total data NON-RUSUN: {len(df_non_rusun)}")
    print("Kolom Kode_8_Digit, Bulan, Tahun dikosongkan karena bukan transaksi SETORTUNAI")
else:
    print("ℹ️ Tidak ada data NON-RUSUN")

# Tampilkan sample dari kedua dataset
print(f"\n🌟 === SAMPLE DATA SETORTUNAI ===")
if len(df_setortunai) > 0:
    display(df_setortunai[['Narasi', 'Kode_8_Digit', 'Bulan', 'Tahun']].head(10))

print(f"\n📝 === SAMPLE DATA NON-RUSUN ===")
if len(df_non_rusun) > 0:
    display(df_non_rusun[['Narasi', 'Kode_8_Digit', 'Bulan', 'Tahun']].head(5))

# Summary akhir
print(f"\n📈 === RINGKASAN PEMISAHAN DATA ===")
print(f"✅ Data SETORTUNAI (diproses): {len(df_setortunai)} baris")
print(f"📄 Data NON-RUSUN (tidak diproses): {len(df_non_rusun)} baris")
print(f"📊 Total keseluruhan: {len(df_setortunai) + len(df_non_rusun)} baris")

if len(df_setortunai) > 0:
    akurasi_setortunai = [
        kode_setortunai/total_setortunai*100,
        bulan_setortunai/total_setortunai*100,
        tahun_setortunai/total_setortunai*100
    ]
    print(f"\n🎯 AKURASI EKSTRAKSI DATA SETORTUNAI:")
    print(f"Kode: {akurasi_setortunai[0]:.1f}% | Bulan: {akurasi_setortunai[1]:.1f}% | Tahun: {akurasi_setortunai[2]:.1f}%")

🔄 === REVISI REGEX UNTUK DATA SETORTUNAI ===
Total data: 114
Data dengan SETORTUNAI: 107 baris
Data NON-RUSUN (tanpa SETORTUNAI): 7 baris

🔍 === PROSES DATA SETORTUNAI ===
📊 HASIL EKSTRAKSI DATA SETORTUNAI:
Total data SETORTUNAI: 107
🔢 Kode 8 Digit: 103 (96.3%)
📅 Bulan: 96 (89.7%)
📆 Tahun: 95 (88.8%)
✅ Data Lengkap: 93 (86.9%)

📋 === PROSES DATA NON-RUSUN ===
📊 DATA NON-RUSUN:
Total data NON-RUSUN: 7
Kolom Kode_8_Digit, Bulan, Tahun dikosongkan karena bukan transaksi SETORTUNAI

🌟 === SAMPLE DATA SETORTUNAI ===


,Narasi,Kode_8_Digit,Bulan,Tahun
0,0023-800476-360/L704-0901SETORTUNAI/MARTINAVENTYDEYRUSUN03020308/BIII8/MARTINA/JUNI25,03020308,Juni,2025
1,0023-800474-360/L778-0535SETORTUNAI/SUSANROSADIQQMUHAMMA01010205/SUSANROSADI/AII5/JUNI2025,01010205,Juni,2025
2,0175-800459-360/K565-0175SETORTUNAI/IIMROSMINIRSNCIBEUREUM/02040203/JUHANA/DII3/MEI2025,02040203,Mei,2025
3,0262-800459-360/J580-0262SETORTUNAI/RINA,,,
4,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/DES24RUSUN/02040104/DUDE/DI4/DES24,02040104,Desember,2024
5,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/JAN25RUSUN/02040104/DUDE/DI4/JAN25,02040104,Januari,2025
6,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/FEB25RUSUN/02040104/DUDE/DI4/FEB25,02040104,Februari,2025
7,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/JUN25RUSUN/02040104/DUDE/DI4/JUN25,02040104,Juni,2025
8,0262-800459-360/J580-0262SETORTUNAI/RIFKYIININDRIANI/03030518/JUNI2025IININDRIANI/03030518/JUNI2025,03030518,Juni,2025
9,0023-800474-360/L778-0535SETORTUNAI/YUSEP,,,



📝 === SAMPLE DATA NON-RUSUN ===


,Narasi,Kode_8_Digit,Bulan,Tahun
29,F297/PELIMPRETRIBUSITGL010725U/REKKASDACIMAHI0230270000025DISPERKIM,,,
30,POTONGANSEWAHUNIARUSUNAWA2025,,,
60,TRFDARIBPPDPKPUPTDRNOREK0131960867001Potongansewarusunawa,,,
61,F297/PELIMPRETRIBUSITGL020725U/REKKASDACIMAHI0230270000025DISPERKIM,,,
87,F297/PELIMPRETRIBUSITGL030725U/REKKASDACIMAHI0230270000025DISPERKIM,,,



📈 === RINGKASAN PEMISAHAN DATA ===
✅ Data SETORTUNAI (diproses): 107 baris
📄 Data NON-RUSUN (tidak diproses): 7 baris
📊 Total keseluruhan: 114 baris

🎯 AKURASI EKSTRAKSI DATA SETORTUNAI:
Kode: 96.3% | Bulan: 89.7% | Tahun: 88.8%


In [4]:
# POST-PROCESSING: FILTERING DATA SETORTUNAI YANG TIDAK LENGKAP
print("🔄 === POST-PROCESSING: FILTERING DATA TIDAK LENGKAP ===")

# Backup data asli sebelum filtering
df_setortunai_original = df_setortunai.copy()
df_non_rusun_original = df_non_rusun.copy()

print(f"📊 STATUS SEBELUM FILTERING:")
print(f"├── Data SETORTUNAI: {len(df_setortunai)} baris")
print(f"└── Data NON-RUSUN: {len(df_non_rusun)} baris")

# Identifikasi data SETORTUNAI yang tidak lengkap
# Kriteria: kode 8 digit, bulan, atau tahun ada yang kosong
incomplete_mask = (
    (df_setortunai['Kode_8_Digit'] == '') |
    (df_setortunai['Bulan'] == '') |
    (df_setortunai['Tahun'] == '')
)

# Data SETORTUNAI yang tidak lengkap akan dipindah ke NON-RUSUN
df_setortunai_incomplete = df_setortunai[incomplete_mask].copy()
df_setortunai_complete = df_setortunai[~incomplete_mask].copy()

# Kosongkan kolom ekstraksi untuk data yang dipindah ke NON-RUSUN
# dan tambahkan kolom keterangan
if len(df_setortunai_incomplete) > 0:
    # Buat kolom keterangan berdasarkan field yang kosong
    keterangan_list = []
    
    for idx, row in df_setortunai_incomplete.iterrows():
        missing_fields = []
        
        if row['Kode_8_Digit'] == '':
            missing_fields.append('Kode 8 Digit')
        if row['Bulan'] == '':
            missing_fields.append('Bulan')
        if row['Tahun'] == '':
            missing_fields.append('Tahun')
        
        if missing_fields:
            keterangan = f"Kekurangan: {', '.join(missing_fields)}"
        else:
            keterangan = "Data tidak lengkap"
        
        keterangan_list.append(keterangan)
    
    # Tambahkan kolom keterangan
    df_setortunai_incomplete['Keterangan'] = keterangan_list
    
    # Kosongkan kolom ekstraksi setelah keterangan dibuat
    df_setortunai_incomplete['Kode_8_Digit'] = ''
    df_setortunai_incomplete['Bulan'] = ''
    df_setortunai_incomplete['Tahun'] = ''
else:
    # Jika tidak ada data yang dipindah, buat DataFrame kosong dengan kolom keterangan
    df_setortunai_incomplete['Keterangan'] = []

# Untuk data non-rusun asli, tambahkan kolom keterangan yang kosong
if len(df_non_rusun) > 0:
    df_non_rusun['Keterangan'] = 'Bukan transaksi SETORTUNAI'

# Gabungkan data tidak lengkap ke NON-RUSUN
df_non_rusun_new = pd.concat([df_non_rusun, df_setortunai_incomplete], ignore_index=True)

# Update DataFrame utama
df_setortunai = df_setortunai_complete.copy()
df_non_rusun = df_non_rusun_new.copy()

# Statistik setelah filtering
print(f"\n📊 STATUS SETELAH FILTERING:")
print(f"├── Data SETORTUNAI (lengkap): {len(df_setortunai)} baris")
print(f"├── Data NON-RUSUN (gabungan): {len(df_non_rusun)} baris")
print(f"└── Data dipindah (tidak lengkap): {len(df_setortunai_incomplete)} baris")

# Analisis perubahan
print(f"\n🔍 === ANALISIS PERUBAHAN ===")
if len(df_setortunai_incomplete) > 0:
    print(f"Data SETORTUNAI yang dipindah ke NON-RUSUN:")
    
    # Analisis jenis ketidaklengkapan
    missing_kode = (df_setortunai_original['Kode_8_Digit'] == '').sum()
    missing_bulan = (df_setortunai_original['Bulan'] == '').sum()
    missing_tahun = (df_setortunai_original['Tahun'] == '').sum()
    
    print(f"├── Tidak ada kode 8 digit: {missing_kode} transaksi")
    print(f"├── Tidak ada bulan: {missing_bulan} transaksi")
    print(f"├── Tidak ada tahun: {missing_tahun} transaksi")
    print(f"└── Total dipindah: {len(df_setortunai_incomplete)} transaksi")
    
    # Sample data yang dipindah
    print(f"\n📋 CONTOH DATA YANG DIPINDAH:")
    sample_moved = df_setortunai_incomplete[['Narasi', 'Kode_8_Digit', 'Bulan', 'Tahun', 'Keterangan']].head(5)
    display(sample_moved)
    
    # Analisis distribusi keterangan
    keterangan_dist = df_setortunai_incomplete['Keterangan'].value_counts()
    print(f"\n📊 DISTRIBUSI KETERANGAN:")
    for keterangan, count in keterangan_dist.items():
        print(f"  - {keterangan}: {count} transaksi")
else:
    print("✅ Tidak ada data yang perlu dipindah - semua data SETORTUNAI sudah lengkap!")

# Statistik final data SETORTUNAI yang tersisa
if len(df_setortunai) > 0:
    total_lengkap = len(df_setortunai)
    
    print(f"\n✅ === DATA SETORTUNAI FINAL (LENGKAP) ===")
    print(f"├── Total data lengkap: {total_lengkap}")
    print(f"├── Kode 8 digit: {total_lengkap} (100.0%)")
    print(f"├── Bulan: {total_lengkap} (100.0%)")
    print(f"├── Tahun: {total_lengkap} (100.0%)")
    print(f"└── Data lengkap: {total_lengkap} (100.0%)")
    
    # Sample data lengkap
    print(f"\n🌟 SAMPLE DATA SETORTUNAI LENGKAP:")
    sample_complete = df_setortunai.head(5)
    display(sample_complete)

# Tampilkan sample data NON-RUSUN dengan keterangan
print(f"\n📄 === SAMPLE DATA NON-RUSUN (DENGAN KETERANGAN) ===")
if len(df_non_rusun) > 0:
    sample_non_rusun = df_non_rusun.head(5)
    display(sample_non_rusun)

# Ringkasan perubahan
persentase_dipindah = (len(df_setortunai_incomplete) / len(df_setortunai_original)) * 100 if len(df_setortunai_original) > 0 else 0
persentase_tersisa = (len(df_setortunai) / len(df_setortunai_original)) * 100 if len(df_setortunai_original) > 0 else 0

print(f"\n📈 === RINGKASAN FILTERING ===")
print(f"├── Data SETORTUNAI asli: {len(df_setortunai_original)} baris")
print(f"├── Data dipindah ke NON-RUSUN: {len(df_setortunai_incomplete)} baris ({persentase_dipindah:.1f}%)")
print(f"├── Data SETORTUNAI tersisa: {len(df_setortunai)} baris ({persentase_tersisa:.1f}%)")
print(f"└── Total NON-RUSUN: {len(df_non_rusun)} baris")

# Validasi tidak ada data yang hilang
total_after = len(df_setortunai) + len(df_non_rusun)
total_before = len(df_setortunai_original) + len(df_non_rusun_original)

if total_after == total_before:
    print(f"\n✅ VALIDASI: Total data konsisten ({total_before} → {total_after})")
else:
    print(f"\n❌ WARNING: Total data tidak konsisten ({total_before} → {total_after})")

print(f"\n🎯 Hasil: Semua data SETORTUNAI sekarang memiliki kode 8 digit, bulan, dan tahun yang lengkap!")

🔄 === POST-PROCESSING: FILTERING DATA TIDAK LENGKAP ===
📊 STATUS SEBELUM FILTERING:
├── Data SETORTUNAI: 107 baris
└── Data NON-RUSUN: 7 baris

📊 STATUS SETELAH FILTERING:
├── Data SETORTUNAI (lengkap): 93 baris
├── Data NON-RUSUN (gabungan): 21 baris
└── Data dipindah (tidak lengkap): 14 baris

🔍 === ANALISIS PERUBAHAN ===
Data SETORTUNAI yang dipindah ke NON-RUSUN:
├── Tidak ada kode 8 digit: 4 transaksi
├── Tidak ada bulan: 11 transaksi
├── Tidak ada tahun: 12 transaksi
└── Total dipindah: 14 transaksi

📋 CONTOH DATA YANG DIPINDAH:


,Narasi,Kode_8_Digit,Bulan,Tahun,Keterangan
3,0262-800459-360/J580-0262SETORTUNAI/RINA,,,,"Kekurangan: Kode 8 Digit, Bulan, Tahun"
9,0023-800474-360/L778-0535SETORTUNAI/YUSEP,,,,"Kekurangan: Kode 8 Digit, Bulan, Tahun"
23,0023-800460-360/H202-0023SETORTUNAI/MROBIULPBYRUUSNLEUWIGAJAHFERRYDIANAAIV15APRIL25,,,,Kekurangan: Kode 8 Digit
31,0485-800460-360/YB43-0485SETORTUNAI/DODIALFIANRUSUNAWACIBEUREUMID02040302RUSUNAWACIBEUREUMID02040302,,,,"Kekurangan: Bulan, Tahun"
43,0439-800459-360/M159-0439SETORTUNAI/HENDRAPBY/02020504/HENDRA/BV4@362.100PBY/02020504/HENDRA/BV4@362.100,,,,"Kekurangan: Bulan, Tahun"



📊 DISTRIBUSI KETERANGAN:
  - Kekurangan: Bulan, Tahun: 9 transaksi
  - Kekurangan: Kode 8 Digit, Bulan, Tahun: 2 transaksi
  - Kekurangan: Kode 8 Digit: 2 transaksi
  - Kekurangan: Tahun: 1 transaksi

✅ === DATA SETORTUNAI FINAL (LENGKAP) ===
├── Total data lengkap: 93
├── Kode 8 digit: 93 (100.0%)
├── Bulan: 93 (100.0%)
├── Tahun: 93 (100.0%)
└── Data lengkap: 93 (100.0%)

🌟 SAMPLE DATA SETORTUNAI LENGKAP:


,No,Posting Date,Effective Date,Narasi,Debit Transaction,Credit Transaction,Balance,Kode_8_Digit,Bulan,Tahun
0,1,01/07/2025,01/07/2025,0023-800476-360/L704-0901SETORTUNAI/MARTINAVENTYDEYRUSUN03020308/BIII8/MARTINA/JUNI25,,430000,430000,03020308,Juni,2025
1,2,01/07/2025,01/07/2025,0023-800474-360/L778-0535SETORTUNAI/SUSANROSADIQQMUHAMMA01010205/SUSANROSADI/AII5/JUNI2025,,402900,832900,01010205,Juni,2025
2,3,01/07/2025,01/07/2025,0175-800459-360/K565-0175SETORTUNAI/IIMROSMINIRSNCIBEUREUM/02040203/JUHANA/DII3/MEI2025,,456450,1289350,02040203,Mei,2025
4,5,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/DES24RUSUN/02040104/DUDE/DI4/DES24,,462500,2205750,02040104,Desember,2024
5,6,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQRAISYADARARUSUN/02040104/DUDE/DI4/JAN25RUSUN/02040104/DUDE/DI4/JAN25,,462500,2668250,02040104,Januari,2025



📄 === SAMPLE DATA NON-RUSUN (DENGAN KETERANGAN) ===


,No,Posting Date,Effective Date,Narasi,Debit Transaction,Credit Transaction,Balance,Kode_8_Digit,Bulan,Tahun,Keterangan
0,30,01/07/2025,01/07/2025,F297/PELIMPRETRIBUSITGL010725U/REKKASDACIMAHI0230270000025DISPERKIM,"12,059,950.00",,,,,,Bukan transaksi SETORTUNAI
1,31,01/07/2025,01/07/2025,POTONGANSEWAHUNIARUSUNAWA2025,,1665000,1665000,,,,Bukan transaksi SETORTUNAI
2,61,02/07/2025,02/07/2025,TRFDARIBPPDPKPUPTDRNOREK0131960867001Potongansewarusunawa,,3050000,15935750,,,,Bukan transaksi SETORTUNAI
3,62,02/07/2025,02/07/2025,F297/PELIMPRETRIBUSITGL020725U/REKKASDACIMAHI0230270000025DISPERKIM,"15,935,750.00",,,,,,Bukan transaksi SETORTUNAI
4,88,03/07/2025,03/07/2025,F297/PELIMPRETRIBUSITGL030725U/REKKASDACIMAHI0230270000025DISPERKIM,"9,718,950.00",,,,,,Bukan transaksi SETORTUNAI



📈 === RINGKASAN FILTERING ===
├── Data SETORTUNAI asli: 107 baris
├── Data dipindah ke NON-RUSUN: 14 baris (13.1%)
├── Data SETORTUNAI tersisa: 93 baris (86.9%)
└── Total NON-RUSUN: 21 baris

✅ VALIDASI: Total data konsisten (114 → 114)

🎯 Hasil: Semua data SETORTUNAI sekarang memiliki kode 8 digit, bulan, dan tahun yang lengkap!


In [5]:
# === EXTRACT DATA DARI EXCEL MASTER BERDASARKAN KODE 8 DIGIT ===
print("📤 === EXTRACT DATA DARI EXCEL MASTER ===")

import openpyxl
from openpyxl import load_workbook
import os
from datetime import datetime

# Fungsi untuk mendapatkan mapping kolom berdasarkan bulan (untuk extract data)
def get_month_columns_extract(bulan):
    """
    Mengembalikan kolom I, J, K, L untuk setiap bulan
    Mapping dengan jarak 10 kolom per bulan
    """
    # Mapping kolom untuk extract data (I, J, K, L dan seterusnya dengan jarak 10 kolom)
    month_mapping = {
        'Januari': ['I', 'J', 'K', 'L'],           # Base columns
        'Februari': ['S', 'T', 'U', 'V'],         # +10 kolom dari Januari
        'Maret': ['AC', 'AD', 'AE', 'AF'],        # +10 kolom dari Februari  
        'April': ['AM', 'AN', 'AO', 'AP'],        # +10 kolom dari Maret
        'Mei': ['AW', 'AX', 'AY', 'AZ'],          # +10 kolom dari April
        'Juni': ['BG', 'BH', 'BI', 'BJ'],         # +10 kolom dari Mei
        'Juli': ['BQ', 'BR', 'BS', 'BT'],         # +10 kolom dari Juni
        'Agustus': ['CA', 'CB', 'CC', 'CD'],      # +10 kolom dari Juli
        'September': ['CK', 'CL', 'CM', 'CN'],    # +10 kolom dari Agustus
        'Oktober': ['CU', 'CV', 'CW', 'CX'],      # +10 kolom dari September
        'November': ['DE', 'DF', 'DG', 'DH'],     # +10 kolom dari Oktober
        'Desember': ['DO', 'DP', 'DQ', 'DR']      # +10 kolom dari November
    }
    return month_mapping.get(bulan, None)

# Fungsi untuk mendapatkan nama sheet berdasarkan 2 digit pertama
def get_sheet_name_extract(kode_2_digit_pertama):
    """
    Mengembalikan nama sheet berdasarkan 2 digit pertama kode
    """
    sheet_mapping = {
        '01': 'CIGUGUR',
        '02': 'MELONG', 
        '03': 'LG '
    }
    return sheet_mapping.get(kode_2_digit_pertama, None)

# Fungsi untuk memecah kode 8 digit
def parse_kode_8_digit_extract(kode):
    """
    Memecah kode 8 digit menjadi 4 bagian: 2-2-2-2
    """
    if len(kode) != 8:
        return None, None, None, None
    
    digit_1_2 = kode[:2]   # Menentukan sheet
    digit_3_4 = kode[2:4]  # Kolom B
    digit_5_6 = kode[4:6]  # Kolom C  
    digit_7_8 = kode[6:8]  # Kolom D
    
    return digit_1_2, digit_3_4, digit_5_6, digit_7_8

# Fungsi untuk mencari row berdasarkan kode di kolom B, C, D
def find_row_by_code_extract(worksheet, digit_3_4, digit_5_6, digit_7_8):
    """
    Mencari row yang sesuai berdasarkan kode di kolom B, C, D
    """
    for row in range(2, worksheet.max_row + 1):  # Mulai dari row 2 (skip header)
        col_b_value = str(worksheet[f'B{row}'].value or '').zfill(2)
        col_c_value = str(worksheet[f'C{row}'].value or '').zfill(2)
        col_d_value = str(worksheet[f'D{row}'].value or '').zfill(2)
        
        if (col_b_value == digit_3_4 and 
            col_c_value == digit_5_6 and 
            col_d_value == digit_7_8):
            return row
    
    return None

# Fungsi untuk memformat tanggal tanpa waktu
def format_date_only(date_value):
    """
    Mengkonversi nilai tanggal dari Excel menjadi string tanggal saja (tanpa waktu 00:00:00)
    """
    if not date_value or date_value == '':
        return ''
    
    # Jika sudah berupa datetime object
    if isinstance(date_value, datetime):
        return date_value.strftime('%Y-%m-%d')
    
    # Jika berupa string yang mengandung datetime
    if isinstance(date_value, str):
        try:
            # Coba parse berbagai format tanggal
            for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y']:
                try:
                    parsed_date = datetime.strptime(date_value, fmt)
                    return parsed_date.strftime('%Y-%m-%d')
                except:
                    continue
        except:
            pass
    
    # Jika gagal parsing, return as string
    return str(date_value)

# Fungsi utama untuk extract data dari Excel
def extract_data_from_excel(df_data):
    """
    Extract data dari Excel master berdasarkan tahun, kode, dan bulan
    Mengembalikan DataFrame dengan 4 kolom tambahan
    """
    print(f"\n🔍 === ANALISIS DATA UNTUK EXTRACT ===")
    
    # Buat copy dari dataframe input
    result_df = df_data.copy()
    
    # Inisialisasi kolom baru
    result_df['Nama Penghuni'] = ''
    result_df['Tanggal Perjanjian Sewa'] = ''
    result_df['Sewa Hunian'] = ''
    result_df['Sewa Lahan Lantai 1'] = ''
    
    # Reorder kolom agar kolom baru berada di sebelah kiri Kode_8_Digit
    cols = result_df.columns.tolist()
    kode_idx = cols.index('Kode_8_Digit')
    
    # Sisipkan kolom baru sebelum Kode_8_Digit
    new_cols = (cols[:kode_idx] + 
                ['Nama Penghuni', 'Tanggal Perjanjian Sewa', 'Sewa Hunian', 'Sewa Lahan Lantai 1'] +
                cols[kode_idx:])
    
    # Drop kolom baru yang sudah ada di posisi lama
    for col in ['Nama Penghuni', 'Tanggal Perjanjian Sewa', 'Sewa Hunian', 'Sewa Lahan Lantai 1']:
        if col in cols[kode_idx:]:
            new_cols.remove(col)
    
    result_df = result_df[new_cols]
    
    # Group data berdasarkan tahun
    data_by_year = result_df.groupby('Tahun')
    
    results = {
        'success': 0,
        'failed': 0,
        'errors': []
    }
    
    for tahun, group_data in data_by_year:
        if not tahun or tahun == '':
            continue
            
        print(f"\n📅 MEMPROSES TAHUN: {tahun}")
        print(f"   Data: {len(group_data)} baris")
        
        # Tentukan file Excel berdasarkan tahun
        if tahun == '2024':
            excel_file = 'Master Data/Master INPUT 2024 FIX.xlsx'
        elif tahun == '2025':
            excel_file = 'Master Data/Master INPUT 2025.xlsx'
        else:
            print(f"   ❌ Tahun {tahun} tidak didukung")
            continue
        
        # Cek apakah file exists
        if not os.path.exists(excel_file):
            print(f"   ❌ File tidak ditemukan: {excel_file}")
            results['errors'].append(f"File tidak ditemukan: {excel_file}")
            continue
        
        try:
            # Load workbook
            print(f"   📂 Membuka file: {excel_file}")
            workbook = load_workbook(excel_file, data_only=True)  # data_only=True untuk mendapatkan nilai, bukan formula
            
            # Process setiap baris data untuk tahun ini
            for idx, row in group_data.iterrows():
                kode_8_digit = str(row['Kode_8_Digit'])
                bulan = row['Bulan']
                
                # Validasi data
                if len(kode_8_digit) != 8:
                    print(f"   ❌ Kode tidak valid: {kode_8_digit}")
                    results['failed'] += 1
                    continue
                
                # Parse kode 8 digit
                digit_1_2, digit_3_4, digit_5_6, digit_7_8 = parse_kode_8_digit_extract(kode_8_digit)
                
                # Tentukan sheet
                sheet_name = get_sheet_name_extract(digit_1_2)
                if not sheet_name:
                    print(f"   ❌ Sheet tidak ditemukan untuk kode: {digit_1_2}")
                    results['failed'] += 1
                    continue
                
                # Cek apakah sheet ada
                if sheet_name not in workbook.sheetnames:
                    print(f"   ❌ Sheet '{sheet_name}' tidak ada di file")
                    results['failed'] += 1
                    continue
                
                worksheet = workbook[sheet_name]
                
                # Cari row berdasarkan kode
                target_row = find_row_by_code_extract(worksheet, digit_3_4, digit_5_6, digit_7_8)
                if not target_row:
                    print(f"   ❌ Row tidak ditemukan untuk kode: {kode_8_digit}")
                    results['failed'] += 1
                    continue
                
                # Dapatkan kolom berdasarkan bulan
                month_cols = get_month_columns_extract(bulan)
                if not month_cols:
                    print(f"   ❌ Bulan tidak valid: {bulan}")
                    results['failed'] += 1
                    continue
                
                # Extract data dari 4 kolom
                try:
                    nama_penghuni = worksheet[f'{month_cols[0]}{target_row}'].value or ''
                    tanggal_perjanjian_raw = worksheet[f'{month_cols[1]}{target_row}'].value or ''
                    sewa_hunian = worksheet[f'{month_cols[2]}{target_row}'].value or ''
                    sewa_lahan = worksheet[f'{month_cols[3]}{target_row}'].value or ''
                    
                    # Format tanggal tanpa waktu
                    tanggal_perjanjian = format_date_only(tanggal_perjanjian_raw)
                    
                    # Update DataFrame result
                    result_df.loc[idx, 'Nama Penghuni'] = str(nama_penghuni)
                    result_df.loc[idx, 'Tanggal Perjanjian Sewa'] = tanggal_perjanjian
                    result_df.loc[idx, 'Sewa Hunian'] = str(sewa_hunian)
                    result_df.loc[idx, 'Sewa Lahan Lantai 1'] = str(sewa_lahan)
                    
                    print(f"   ✅ Extract berhasil: {kode_8_digit} | {bulan} | Row {target_row} | Sheet {sheet_name}")
                    results['success'] += 1
                    
                except Exception as e:
                    print(f"   ❌ Error extract: {kode_8_digit} - {str(e)}")
                    results['failed'] += 1
                    results['errors'].append(f"Error extract {kode_8_digit}: {str(e)}")
            
            workbook.close()
            
        except Exception as e:
            print(f"   ❌ Error membuka file: {str(e)}")
            results['errors'].append(f"Error file {excel_file}: {str(e)}")
    
    return result_df, results

# Jalankan fungsi extract data
print("🚀 === MEMULAI EXTRACT DATA DARI EXCEL MASTER ===")

# Pastikan ada data SETORTUNAI yang lengkap
if 'df_setortunai' in locals() and len(df_setortunai) > 0:
    # Filter data yang memiliki semua field yang diperlukan untuk extract
    valid_data = df_setortunai[
        (df_setortunai['Kode_8_Digit'] != '') & 
        (df_setortunai['Bulan'] != '') & 
        (df_setortunai['Tahun'] != '')
    ].copy()
    
    print(f"📊 DATA YANG AKAN DIEXTRACT:")
    print(f"├── Total data SETORTUNAI: {len(df_setortunai)}")
    print(f"├── Data valid untuk extract: {len(valid_data)}")
    print(f"└── Data tidak valid: {len(df_setortunai) - len(valid_data)}")
    
    if len(valid_data) > 0:
        # Tampilkan mapping kolom extract
        print(f"\n📅 === MAPPING KOLOM EXTRACT ===")
        print("Per Bulan (4 kolom: Nama Penghuni, Tanggal Perjanjian, Sewa Hunian, Sewa Lahan):")
        month_mappings_extract = [
            ("Januari", "I, J, K, L"),
            ("Februari", "S, T, U, V"),
            ("Maret", "AC, AD, AE, AF"),
            ("April", "AM, AN, AO, AP"),
            ("Mei", "AW, AX, AY, AZ"),
            ("Juni", "BG, BH, BI, BJ"),
            ("Juli", "BQ, BR, BS, BT"),
            ("Agustus", "CA, CB, CC, CD"),
            ("September", "CK, CL, CM, CN"),
            ("Oktober", "CU, CV, CW, CX"),
            ("November", "DE, DF, DG, DH"),
            ("Desember", "DO, DP, DQ, DR")
        ]
        
        for bulan, cols in month_mappings_extract:
            print(f"├── {bulan:10s}: {cols}")
        
        # Tampilkan distribusi data
        print(f"\n📈 DISTRIBUSI DATA:")
        print("Per Tahun:")
        year_dist = valid_data['Tahun'].value_counts()
        for year, count in year_dist.items():
            print(f"  - {year}: {count} data")
        
        print("\nPer Sheet (2 digit pertama):")
        valid_data['Sheet_Code'] = valid_data['Kode_8_Digit'].str[:2]
        sheet_dist = valid_data['Sheet_Code'].value_counts()
        for code, count in sheet_dist.items():
            sheet_name = get_sheet_name_extract(code)
            print(f"  - {code} ({sheet_name}): {count} data")
        
        # Sample data sebelum extract
        print(f"\n🌟 SAMPLE DATA SEBELUM EXTRACT:")
        sample_cols = ['Kode_8_Digit', 'Bulan', 'Tahun']
        display(valid_data[sample_cols].head(5))
        
        # Jalankan extract
        print(f"\n🔄 MEMULAI PROSES EXTRACT...")
        df_with_extract, results = extract_data_from_excel(valid_data)
        
        # Tampilkan hasil
        print(f"\n🎯 === HASIL EXTRACT DATA ===")
        print(f"✅ Berhasil: {results['success']} data")
        print(f"❌ Gagal: {results['failed']} data")
        
        if results['errors']:
            print(f"\n🚨 ERROR YANG TERJADI:")
            for i, error in enumerate(results['errors'][:5], 1):  # Tampilkan max 5 error
                print(f"  {i}. {error}")
            if len(results['errors']) > 5:
                print(f"  ... dan {len(results['errors']) - 5} error lainnya")
        
        if results['success'] > 0:
            # Update dataframe utama
            df_setortunai_with_extract = df_with_extract.copy()
            
            # Tampilkan sample hasil extract
            print(f"\n🌟 SAMPLE DATA HASIL EXTRACT:")
            extract_cols = ['Nama Penghuni', 'Tanggal Perjanjian Sewa', 'Sewa Hunian', 'Sewa Lahan Lantai 1', 'Kode_8_Digit', 'Bulan', 'Tahun']
            sample_result = df_setortunai_with_extract[extract_cols].head(10)
            display(sample_result)
            
            # Statistik data yang berhasil diextract
            success_mask = (df_setortunai_with_extract['Nama Penghuni'] != '') | \
                          (df_setortunai_with_extract['Tanggal Perjanjian Sewa'] != '') | \
                          (df_setortunai_with_extract['Sewa Hunian'] != '') | \
                          (df_setortunai_with_extract['Sewa Lahan Lantai 1'] != '')
            
            success_count = success_mask.sum()
            print(f"\n📊 STATISTIK HASIL:")
            print(f"├── Data berhasil diextract: {success_count}/{len(df_setortunai_with_extract)}")
            print(f"├── Persentase berhasil: {success_count/len(df_setortunai_with_extract)*100:.1f}%")
            
            # Analisis per kolom
            nama_filled = (df_setortunai_with_extract['Nama Penghuni'] != '').sum()
            tanggal_filled = (df_setortunai_with_extract['Tanggal Perjanjian Sewa'] != '').sum()
            sewa_hunian_filled = (df_setortunai_with_extract['Sewa Hunian'] != '').sum()
            sewa_lahan_filled = (df_setortunai_with_extract['Sewa Lahan Lantai 1'] != '').sum()
            
            print(f"\n📋 ANALISIS PER KOLOM:")
            print(f"├── Nama Penghuni: {nama_filled} data ({nama_filled/len(df_setortunai_with_extract)*100:.1f}%)")
            print(f"├── Tanggal Perjanjian: {tanggal_filled} data ({tanggal_filled/len(df_setortunai_with_extract)*100:.1f}%)")
            print(f"├── Sewa Hunian: {sewa_hunian_filled} data ({sewa_hunian_filled/len(df_setortunai_with_extract)*100:.1f}%)")
            print(f"└── Sewa Lahan Lantai 1: {sewa_lahan_filled} data ({sewa_lahan_filled/len(df_setortunai_with_extract)*100:.1f}%)")
            
            print(f"\n🎉 EXTRACT DATA SELESAI!")
            print(f"📊 DataFrame df_setortunai_with_extract tersedia dengan {len(df_setortunai_with_extract)} baris dan 4 kolom tambahan")
        
    else:
        print("❌ Tidak ada data valid untuk diextract")
        
else:
    print("❌ Data SETORTUNAI tidak tersedia")

📤 === EXTRACT DATA DARI EXCEL MASTER ===
🚀 === MEMULAI EXTRACT DATA DARI EXCEL MASTER ===
📊 DATA YANG AKAN DIEXTRACT:
├── Total data SETORTUNAI: 93
├── Data valid untuk extract: 93
└── Data tidak valid: 0

📅 === MAPPING KOLOM EXTRACT ===
Per Bulan (4 kolom: Nama Penghuni, Tanggal Perjanjian, Sewa Hunian, Sewa Lahan):
├── Januari   : I, J, K, L
├── Februari  : S, T, U, V
├── Maret     : AC, AD, AE, AF
├── April     : AM, AN, AO, AP
├── Mei       : AW, AX, AY, AZ
├── Juni      : BG, BH, BI, BJ
├── Juli      : BQ, BR, BS, BT
├── Agustus   : CA, CB, CC, CD
├── September : CK, CL, CM, CN
├── Oktober   : CU, CV, CW, CX
├── November  : DE, DF, DG, DH
├── Desember  : DO, DP, DQ, DR

📈 DISTRIBUSI DATA:
Per Tahun:
  - 2025: 89 data
  - 2024: 4 data

Per Sheet (2 digit pertama):
  - 02 (MELONG): 44 data
  - 01 (CIGUGUR): 30 data
  - 03 (LG ): 19 data

🌟 SAMPLE DATA SEBELUM EXTRACT:


,Kode_8_Digit,Bulan,Tahun
0,03020308,Juni,2025
1,01010205,Juni,2025
2,02040203,Mei,2025
4,02040104,Desember,2024
5,02040104,Januari,2025



🔄 MEMULAI PROSES EXTRACT...

🔍 === ANALISIS DATA UNTUK EXTRACT ===

📅 MEMPROSES TAHUN: 2024
   Data: 4 baris
   📂 Membuka file: Master Data/Master INPUT 2024 FIX.xlsx
   ✅ Extract berhasil: 02040104 | Desember | Row 303 | Sheet MELONG
   ✅ Extract berhasil: 02040316 | September | Row 349 | Sheet MELONG
   ✅ Extract berhasil: 02030405 | November | Row 256 | Sheet MELONG
   ✅ Extract berhasil: 03030211 | November | Row 214 | Sheet LG 

📅 MEMPROSES TAHUN: 2025
   Data: 89 baris
   📂 Membuka file: Master Data/Master INPUT 2025.xlsx
   ✅ Extract berhasil: 02040104 | Desember | Row 303 | Sheet MELONG
   ✅ Extract berhasil: 02040316 | September | Row 349 | Sheet MELONG
   ✅ Extract berhasil: 02030405 | November | Row 256 | Sheet MELONG
   ✅ Extract berhasil: 03030211 | November | Row 214 | Sheet LG 

📅 MEMPROSES TAHUN: 2025
   Data: 89 baris
   📂 Membuka file: Master Data/Master INPUT 2025.xlsx
   ✅ Extract berhasil: 03020308 | Juni | Row 136 | Sheet LG 
   ✅ Extract berhasil: 01010205 | Jun

,Nama Penghuni,Tanggal Perjanjian Sewa,Sewa Hunian,Sewa Lahan Lantai 1,Kode_8_Digit,Bulan,Tahun
0,Martina,2022-09-20,385000,45000,03020308,Juni,2025
1,Susan Rosadi,2023-08-18,350000,45000,01010205,Juni,2025
2,Juhana,2022-09-01,425000,22500,02040203,Mei,2025
4,Dude Memed,2020-04-14,440000,22500,02040104,Desember,2024
5,Dude Memed,2020-04-14,440000,22500,02040104,Januari,2025
6,Dude Memed,2020-04-14,440000,22500,02040104,Februari,2025
7,Dude Memed,2020-04-14,440000,22500,02040104,Juni,2025
8,Iin Indriani,2022-12-22,355000,,03030518,Juni,2025
10,Novi Diana,2021-03-08,365000,22500,01030106,Mei,2025
11,Novi Diana,2021-03-08,365000,22500,01030106,Juni,2025



📊 STATISTIK HASIL:
├── Data berhasil diextract: 93/93
├── Persentase berhasil: 100.0%

📋 ANALISIS PER KOLOM:
├── Nama Penghuni: 93 data (100.0%)
├── Tanggal Perjanjian: 93 data (100.0%)
├── Sewa Hunian: 93 data (100.0%)
└── Sewa Lahan Lantai 1: 81 data (87.1%)

🎉 EXTRACT DATA SELESAI!
📊 DataFrame df_setortunai_with_extract tersedia dengan 93 baris dan 4 kolom tambahan


In [6]:
# Tambah kolom berdasarkan parsing kode 8 digit dan kolom Denda
def convert_to_numeric(value):
    if pd.isna(value) or value == '' or value == 'None':
        return 0
    try:
        return float(str(value).replace(',', ''))
    except:
        return 0

def parse_kode_8_digit(kode):
    if pd.isna(kode) or len(str(kode)) != 8:
        return '', '', '', ''
    
    kode_str = str(kode)
    
    # 2 digit pertama - Rusunawa
    rusunawa_map = {'01': 'Cigugur Tengah', '02': 'Cibeureum', '03': 'Leuwigajah'}
    rusunawa = rusunawa_map.get(kode_str[:2], '')
    
    # 2 digit kedua - Gedung (A, B, C, ...)
    gedung_num = int(kode_str[2:4]) if kode_str[2:4].isdigit() else 0
    gedung = chr(64 + gedung_num) if 1 <= gedung_num <= 26 else ''
    
    # 2 digit ketiga - Lantai (I, II, III, ...)
    lantai_num = int(kode_str[4:6]) if kode_str[4:6].isdigit() else 0
    lantai_map = {1:'I', 2:'II', 3:'III', 4:'IV', 5:'V', 6:'VI', 7:'VII', 8:'VIII', 9:'IX', 10:'X'}
    lantai = lantai_map.get(lantai_num, str(lantai_num) if lantai_num > 0 else '')
    
    # 2 digit terakhir - No Hunian
    no_hunian = kode_str[6:8]
    
    return rusunawa, gedung, lantai, no_hunian

# Parse kode 8 digit
parsing_results = df_setortunai_with_extract['Kode_8_Digit'].apply(parse_kode_8_digit)
df_setortunai_with_extract['Rusunawa'] = parsing_results.apply(lambda x: x[0])
df_setortunai_with_extract['Gedung'] = parsing_results.apply(lambda x: x[1])
df_setortunai_with_extract['Lantai'] = parsing_results.apply(lambda x: x[2])
df_setortunai_with_extract['No Hunian'] = parsing_results.apply(lambda x: x[3])

# Hitung kolom Denda = Credit Transaction - Sewa Hunian - Sewa Lahan
df_setortunai_with_extract['Credit_Numeric'] = df_setortunai_with_extract['Credit Transaction'].apply(convert_to_numeric)
df_setortunai_with_extract['Sewa_Hunian_Numeric'] = df_setortunai_with_extract['Sewa Hunian'].apply(convert_to_numeric)
df_setortunai_with_extract['Sewa_Lahan_Numeric'] = df_setortunai_with_extract['Sewa Lahan Lantai 1'].apply(convert_to_numeric)

df_setortunai_with_extract['Denda'] = (df_setortunai_with_extract['Credit_Numeric'] - 
                                       df_setortunai_with_extract['Sewa_Hunian_Numeric'] - 
                                       df_setortunai_with_extract['Sewa_Lahan_Numeric'])

# Hapus kolom sementara dan Sheet_Code
cols_to_drop = ['Credit_Numeric', 'Sewa_Hunian_Numeric', 'Sewa_Lahan_Numeric']
if 'Sheet_Code' in df_setortunai_with_extract.columns:
    cols_to_drop.append('Sheet_Code')

df_final = df_setortunai_with_extract.drop(columns=cols_to_drop)

# Reorder kolom: pindahkan Rusunawa, Gedung, Lantai, No Hunian ke sebelah kiri Nama Penghuni
cols = df_final.columns.tolist()
nama_penghuni_idx = cols.index('Nama Penghuni')

# Kolom yang akan dipindah
cols_to_move = ['Rusunawa', 'Gedung', 'Lantai', 'No Hunian']

# Hapus kolom dari posisi asli
remaining_cols = [col for col in cols if col not in cols_to_move]

# Sisipkan kolom sebelum Nama Penghuni
new_nama_penghuni_idx = remaining_cols.index('Nama Penghuni')
new_cols = (remaining_cols[:new_nama_penghuni_idx] + 
            cols_to_move + 
            remaining_cols[new_nama_penghuni_idx:])

df_final = df_final[new_cols]

# Display dataframe final
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(f"Final DataFrame: {len(df_final)} rows × {len(df_final.columns)} columns")
display(df_final.head(10))



Final DataFrame: 93 rows × 19 columns


,No,Posting Date,Effective Date,Narasi,Debit Transaction,Credit Transaction,Balance,Kode_8_Digit,Bulan,Tahun,Rusunawa,Gedung,Lantai,No Hunian,Nama Penghuni,Tanggal Perjanjian Sewa,Sewa Hunian,Sewa Lahan Lantai 1,Denda
0,1,01/07/2025,01/07/2025,0023-800476-360/L704-0901SETORTUNAI/MARTINAVEN...,,430000,430000,03020308,Juni,2025,Leuwigajah,B,III,08,Martina,2022-09-20,385000,45000,0.0
1,2,01/07/2025,01/07/2025,0023-800474-360/L778-0535SETORTUNAI/SUSANROSAD...,,402900,832900,01010205,Juni,2025,Cigugur Tengah,A,II,05,Susan Rosadi,2023-08-18,350000,45000,7900.0
2,3,01/07/2025,01/07/2025,0175-800459-360/K565-0175SETORTUNAI/IIMROSMINI...,,456450,1289350,02040203,Mei,2025,Cibeureum,D,II,03,Juhana,2022-09-01,425000,22500,8950.0
4,5,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQR...,,462500,2205750,02040104,Desember,2024,Cibeureum,D,I,04,Dude Memed,2020-04-14,440000,22500,0.0
5,6,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQR...,,462500,2668250,02040104,Januari,2025,Cibeureum,D,I,04,Dude Memed,2020-04-14,440000,22500,0.0
6,7,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQR...,,462500,3130750,02040104,Februari,2025,Cibeureum,D,I,04,Dude Memed,2020-04-14,440000,22500,0.0
7,8,01/07/2025,01/07/2025,0169-800459-360/M173-0169SETORTUNAI/TASMALAQQR...,,462500,3593250,02040104,Juni,2025,Cibeureum,D,I,04,Dude Memed,2020-04-14,440000,22500,0.0
8,9,01/07/2025,01/07/2025,0262-800459-360/J580-0262SETORTUNAI/RIFKYIININ...,,355000,3948250,03030518,Juni,2025,Leuwigajah,C,V,18,Iin Indriani,2022-12-22,355000,,0.0
10,11,01/07/2025,01/07/2025,0564-800459-360/K757-0564SETORTUNAI/NANASUMARN...,,410750,4716000,01030106,Mei,2025,Cigugur Tengah,C,I,06,Novi Diana,2021-03-08,365000,22500,23250.0
11,12,01/07/2025,01/07/2025,0564-800459-360/K757-0564SETORTUNAI/NANASUMARN...,,395250,5111250,01030106,Juni,2025,Cigugur Tengah,C,I,06,Novi Diana,2021-03-08,365000,22500,7750.0


In [39]:
# Cek kolom yang ada di df_final dan df_final_with_extract
print("📋 KOLOM YANG TERSEDIA DI df_final:")
if 'df_final' in locals():
    print(f"Total kolom: {len(df_final.columns)}")
    print("Kolom:")
    for i, col in enumerate(df_final.columns, 1):
        print(f"{i:2d}. {col}")
    
    # Tampilkan sample data untuk melihat nilai
    print(f"\n🌟 SAMPLE DATA df_final:")
    display(df_final.head(2))
else:
    print("❌ df_final tidak tersedia")

print("\n" + "="*60)

print("📋 KOLOM YANG TERSEDIA DI df_final_with_extract:")
if 'df_final_with_extract' in locals():
    print(f"Total kolom: {len(df_final_with_extract.columns)}")
    print("Kolom:")
    for i, col in enumerate(df_final_with_extract.columns, 1):
        print(f"{i:2d}. {col}")
    
    # Cek apakah ada kolom Denda
    if 'Denda' in df_final_with_extract.columns:
        print(f"\n✅ Kolom 'Denda' ditemukan!")
        print(f"Sample nilai Denda:")
        sample_denda = df_final_with_extract['Denda'].dropna().head(5)
        for idx, val in sample_denda.items():
            print(f"  - Index {idx}: {val}")
    else:
        print(f"\n❌ Kolom 'Denda' tidak ditemukan!")
        
else:
    print("❌ df_final_with_extract tidak tersedia")

📋 KOLOM YANG TERSEDIA DI df_final:
Total kolom: 22
Kolom:
 1. No
 2. Posting Date
 3. Effective Date
 4. Narasi
 5. Debit Transaction
 6. Credit Transaction
 7. Balance
 8. Kode_8_Digit
 9. Bulan
10. Tahun
11. Rusunawa
12. Gedung
13. Lantai
14. No Hunian
15. Nama Penghuni
16. Tanggal Perjanjian Sewa
17. Sewa Hunian
18. Sewa Lahan Lantai 1
19. Credit_Numeric
20. Sewa_Hunian_Numeric
21. Sewa_Lahan_Numeric
22. Denda

🌟 SAMPLE DATA df_final:


,No,Posting Date,Effective Date,Narasi,Debit Transaction,Credit Transaction,Balance,Kode_8_Digit,Bulan,Tahun,Rusunawa,Gedung,Lantai,No Hunian,Nama Penghuni,Tanggal Perjanjian Sewa,Sewa Hunian,Sewa Lahan Lantai 1,Credit_Numeric,Sewa_Hunian_Numeric,Sewa_Lahan_Numeric,Denda
0,1,01/07/2025,01/07/2025,0023-800476-360/L704-0901SETORTUNAI/MARTINAVEN...,,430000,430000,03020308,Juni,2025,Leuwigajah,B,III,08,Martina,2022-09-20,385000,45000,430000.0,385000.0,45000.0,0.0
1,2,01/07/2025,01/07/2025,0023-800474-360/L778-0535SETORTUNAI/SUSANROSAD...,,402900,832900,01010205,Juni,2025,Cigugur Tengah,A,II,05,Susan Rosadi,2023-08-18,350000,45000,402900.0,350000.0,45000.0,7900.0



📋 KOLOM YANG TERSEDIA DI df_final_with_extract:
❌ df_final_with_extract tidak tersedia


In [38]:
# === INPUT DATA KE EXCEL MASTER BERDASARKAN TAHUN (VERSI TERBARU) ===
print("📥 === INPUT DATA KE EXCEL MASTER (SAVE AS + CEK DATA TERISI + PEWARNAAN) ===")

import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import os
import shutil

# Fungsi untuk mendapatkan mapping kolom berdasarkan bulan (DIPERBAIKI + PEWARNAAN)
def get_month_columns(bulan):
    """
    Mengembalikan kolom posting date, denda, dan range kolom untuk pewarnaan (10 kolom per bulan)
    Mapping dengan jarak 10 kolom per bulan
    """
    month_mapping = {
        'Januari': {'posting_date_cols': ['P', 'Q'], 'denda_col': 'M', 'color_range': ['I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R']},
        'Februari': {'posting_date_cols': ['Z', 'AA'], 'denda_col': 'W', 'color_range': ['S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'AA', 'AB']},
        'Maret': {'posting_date_cols': ['AJ', 'AK'], 'denda_col': 'AG', 'color_range': ['AC', 'AD', 'AE', 'AF', 'AG', 'AH', 'AI', 'AJ', 'AK', 'AL']},
        'April': {'posting_date_cols': ['AT', 'AU'], 'denda_col': 'AQ', 'color_range': ['AM', 'AN', 'AO', 'AP', 'AQ', 'AR', 'AS', 'AT', 'AU', 'AV']},
        'Mei': {'posting_date_cols': ['BD', 'BE'], 'denda_col': 'BA', 'color_range': ['AW', 'AX', 'AY', 'AZ', 'BA', 'BB', 'BC', 'BD', 'BE', 'BF']},
        'Juni': {'posting_date_cols': ['BN', 'BO'], 'denda_col': 'BK', 'color_range': ['BG', 'BH', 'BI', 'BJ', 'BK', 'BL', 'BM', 'BN', 'BO', 'BP']},
        'Juli': {'posting_date_cols': ['BX', 'BY'], 'denda_col': 'BU', 'color_range': ['BQ', 'BR', 'BS', 'BT', 'BU', 'BV', 'BW', 'BX', 'BY', 'BZ']},
        'Agustus': {'posting_date_cols': ['CH', 'CI'], 'denda_col': 'CE', 'color_range': ['CA', 'CB', 'CC', 'CD', 'CE', 'CF', 'CG', 'CH', 'CI', 'CJ']},
        'September': {'posting_date_cols': ['CR', 'CS'], 'denda_col': 'CO', 'color_range': ['CK', 'CL', 'CM', 'CN', 'CO', 'CP', 'CQ', 'CR', 'CS', 'CT']},
        'Oktober': {'posting_date_cols': ['DB', 'DC'], 'denda_col': 'CY', 'color_range': ['CU', 'CV', 'CW', 'CX', 'CY', 'CZ', 'DA', 'DB', 'DC', 'DD']},
        'November': {'posting_date_cols': ['DL', 'DM'], 'denda_col': 'DI', 'color_range': ['DE', 'DF', 'DG', 'DH', 'DI', 'DJ', 'DK', 'DL', 'DM', 'DN']},
        'Desember': {'posting_date_cols': ['DV', 'DW'], 'denda_col': 'DS', 'color_range': ['DO', 'DP', 'DQ', 'DR', 'DS', 'DT', 'DU', 'DV', 'DW', 'DX']}
    }
    return month_mapping.get(bulan, None)

# Fungsi untuk mendapatkan nama sheet berdasarkan 2 digit pertama
def get_sheet_name(kode_2_digit_pertama):
    """
    Mengembalikan nama sheet berdasarkan 2 digit pertama kode
    """
    sheet_mapping = {
        '01': 'CIGUGUR',
        '02': 'MELONG', 
        '03': 'LG '
    }
    return sheet_mapping.get(kode_2_digit_pertama, None)

# Fungsi untuk memecah kode 8 digit
def parse_kode_8_digit(kode):
    """
    Memecah kode 8 digit menjadi 4 bagian: 2-2-2-2
    """
    if len(kode) != 8:
        return None, None, None, None
    
    digit_1_2 = kode[:2]   # Menentukan sheet
    digit_3_4 = kode[2:4]  # Kolom B
    digit_5_6 = kode[4:6]  # Kolom C  
    digit_7_8 = kode[6:8]  # Kolom D
    
    return digit_1_2, digit_3_4, digit_5_6, digit_7_8

# Fungsi untuk mencari row berdasarkan kode di kolom B, C, D
def find_row_by_code(worksheet, digit_3_4, digit_5_6, digit_7_8):
    """
    Mencari row yang sesuai berdasarkan kode di kolom B, C, D
    """
    for row in range(2, worksheet.max_row + 1):  # Mulai dari row 2 (skip header)
        col_b_value = str(worksheet[f'B{row}'].value or '').zfill(2)
        col_c_value = str(worksheet[f'C{row}'].value or '').zfill(2)
        col_d_value = str(worksheet[f'D{row}'].value or '').zfill(2)
        
        if (col_b_value == digit_3_4 and 
            col_c_value == digit_5_6 and 
            col_d_value == digit_7_8):
            return row
    
    return None

# Fungsi untuk cek apakah cell sudah terisi
def is_cell_filled(worksheet, cell_address):
    """
    Cek apakah cell sudah terisi data
    """
    cell_value = worksheet[cell_address].value
    return cell_value is not None and str(cell_value).strip() != ''

# Fungsi untuk membuat file backup dengan save as
def create_backup_file(original_file):
    """
    Membuat backup file dengan timestamp
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path_parts = original_file.split('/')
    filename = path_parts[-1]
    directory = '/'.join(path_parts[:-1])
    
    # Buat nama file backup
    name_without_ext = filename.split('.')[0]
    backup_filename = f"{name_without_ext}_BACKUP_{timestamp}.xlsx"
    backup_path = f"{directory}/{backup_filename}"
    
    # Copy file
    shutil.copy2(original_file, backup_path)
    return backup_path

# Fungsi utama untuk input data ke Excel (VERSI TERBARU)
def input_data_to_excel_v2(df_data):
    """
    Input data ke Excel master berdasarkan tahun, kode, dan bulan
    FITUR BARU:
    - Save as untuk backup file asli
    - Cek data yang sudah terisi dan skip
    - Keterangan untuk data yang di-skip
    """
    print(f"\n🔍 === ANALISIS DATA UNTUK INPUT (V2) ===")
    
    # Group data berdasarkan tahun
    data_by_year = df_data.groupby('Tahun')
    
    results = {
        'success': 0,
        'skipped': 0,
        'failed': 0,
        'errors': [],
        'skipped_details': [],
        'success_details': [],
        'failed_details': []
    }
    
    for tahun, group_data in data_by_year:
        if not tahun or tahun == '':
            continue
            
        print(f"\n📅 MEMPROSES TAHUN: {tahun}")
        print(f"   Data: {len(group_data)} baris")
        
        # Tentukan file Excel berdasarkan tahun
        if tahun == '2024':
            excel_file = 'Master Data/Master INPUT 2024 FIX.xlsx'
        elif tahun == '2025':
            excel_file = 'Master Data/Master INPUT 2025.xlsx'
        else:
            print(f"   ❌ Tahun {tahun} tidak didukung")
            continue
        
        # Cek apakah file exists
        if not os.path.exists(excel_file):
            print(f"   ❌ File tidak ditemukan: {excel_file}")
            results['errors'].append(f"File tidak ditemukan: {excel_file}")
            continue
        
        try:
            # Buat backup file
            print(f"   💾 Membuat backup file...")
            backup_path = create_backup_file(excel_file)
            print(f"   📋 Backup disimpan: {backup_path}")
            
            # Load workbook dari backup
            print(f"   📂 Membuka file backup untuk diedit...")
            workbook = load_workbook(backup_path)
            
            # Process setiap baris data
            for idx, row in group_data.iterrows():
                kode_8_digit = str(row['Kode_8_Digit'])
                bulan = row['Bulan']
                posting_date = row['Posting Date']
                denda_value = row['Denda']  # Menggunakan kolom Denda
                
                # Validasi data
                if len(kode_8_digit) != 8:
                    print(f"   ❌ Kode tidak valid: {kode_8_digit}")
                    results['failed'] += 1
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': 'Kode tidak valid'
                    })
                    continue
                
                # Parse kode 8 digit
                digit_1_2, digit_3_4, digit_5_6, digit_7_8 = parse_kode_8_digit(kode_8_digit)
                
                # Tentukan sheet
                sheet_name = get_sheet_name(digit_1_2)
                if not sheet_name:
                    print(f"   ❌ Sheet tidak ditemukan untuk kode: {digit_1_2}")
                    results['failed'] += 1
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': f'Sheet tidak ditemukan untuk kode: {digit_1_2}'
                    })
                    continue
                
                # Cek apakah sheet ada
                if sheet_name not in workbook.sheetnames:
                    print(f"   ❌ Sheet '{sheet_name}' tidak ada di file")
                    results['failed'] += 1
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': f"Sheet '{sheet_name}' tidak ada di file"
                    })
                    continue
                
                worksheet = workbook[sheet_name]
                
                # Cari row berdasarkan kode
                target_row = find_row_by_code(worksheet, digit_3_4, digit_5_6, digit_7_8)
                if not target_row:
                    print(f"   ❌ Row tidak ditemukan untuk kode: {kode_8_digit}")
                    results['failed'] += 1
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': f'Row tidak ditemukan untuk kode: {kode_8_digit}'
                    })
                    continue
                
                # Dapatkan kolom berdasarkan bulan
                month_cols = get_month_columns(bulan)
                if not month_cols:
                    print(f"   ❌ Bulan tidak valid: {bulan}")
                    results['failed'] += 1
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': f'Bulan tidak valid: {bulan}'
                    })
                    continue
                
                # CEK APAKAH DATA SUDAH TERISI
                posting_col_1 = month_cols['posting_date_cols'][0]
                posting_col_2 = month_cols['posting_date_cols'][1]
                denda_col = month_cols['denda_col']
                
                cell1_address = f'{posting_col_1}{target_row}'
                cell2_address = f'{posting_col_2}{target_row}'
                cell3_address = f'{denda_col}{target_row}'
                
                # Cek apakah salah satu cell sudah terisi
                cell1_filled = is_cell_filled(worksheet, cell1_address)
                cell2_filled = is_cell_filled(worksheet, cell2_address)
                cell3_filled = is_cell_filled(worksheet, cell3_address)
                
                if cell1_filled or cell2_filled or cell3_filled:
                    # Data sudah terisi, skip dan catat
                    skip_reason = f"Data sudah terisi di {sheet_name} row {target_row}"
                    results['skipped'] += 1
                    results['skipped_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Sheet': sheet_name,
                        'Row': target_row,
                        'Reason': skip_reason,
                        'Cell1_Filled': cell1_filled,
                        'Cell2_Filled': cell2_filled,
                        'Cell3_Filled': cell3_filled
                    })
                    print(f"   ⏭️ Skip {kode_8_digit}: {skip_reason}")
                    continue
                
                # Input data ke Excel (jika belum terisi)
                try:
                    # Input posting date ke kolom P dan Q (atau sesuai bulan)
                    worksheet[cell1_address] = posting_date
                    worksheet[cell2_address] = posting_date
                    
                    # Input denda ke kolom denda
                    worksheet[cell3_address] = denda_value
                    
                    # PEWARNAAN CELL YANG BERHASIL DI-INPUT
                    # Warna hijau muda #D8E4BC untuk 10 kolom sesuai bulan
                    fill_color = PatternFill(start_color='D8E4BC', end_color='D8E4BC', fill_type='solid')
                    
                    # Warnai 10 kolom untuk bulan ini
                    color_range = month_cols['color_range']
                    for col in color_range:
                        cell_address = f'{col}{target_row}'
                        worksheet[cell_address].fill = fill_color
                    
                    print(f"   ✅ Input berhasil: {kode_8_digit} | {bulan} | Row {target_row} | Sheet {sheet_name} | Denda: {denda_value}")
                    results['success'] += 1
                    results['success_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Sheet': sheet_name,
                        'Row': target_row,
                        'Denda': denda_value,
                        'Reason': 'Data berhasil diinput ke Excel Master'
                    })
                    
                except Exception as e:
                    print(f"   ❌ Error input: {kode_8_digit} - {str(e)}")
                    results['failed'] += 1
                    results['errors'].append(f"Error input {kode_8_digit}: {str(e)}")
                    results['failed_details'].append({
                        'Kode_8_Digit': kode_8_digit,
                        'Bulan': bulan,
                        'Reason': f'Error input: {str(e)}'
                    })
            
            # Simpan workbook (file backup yang sudah diedit)
            workbook.save(backup_path)
            print(f"   💾 File backup dengan data baru disimpan: {backup_path}")
            
        except Exception as e:
            print(f"   ❌ Error membuka file: {str(e)}")
            results['errors'].append(f"Error file {excel_file}: {str(e)}")
    
    return results

# Jalankan fungsi input data VERSI BARU
print("🚀 === MEMULAI INPUT DATA KE EXCEL MASTER (V2) ===")

# Pastikan ada data SETORTUNAI yang lengkap dengan kolom Denda
if 'df_setortunai_with_extract' in locals() and len(df_setortunai_with_extract) > 0:
    # Filter data yang memiliki semua field yang diperlukan
    valid_data = df_setortunai_with_extract[
        (df_setortunai_with_extract['Kode_8_Digit'] != '') & 
        (df_setortunai_with_extract['Bulan'] != '') & 
        (df_setortunai_with_extract['Tahun'] != '') &
        (df_setortunai_with_extract['Posting Date'].notna()) &
        (df_setortunai_with_extract['Denda'].notna())  # Menggunakan kolom Denda
    ].copy()
    
    print(f"📊 DATA YANG AKAN DIINPUT:")
    print(f"├── Total data SETORTUNAI dengan extract: {len(df_setortunai_with_extract)}")
    print(f"├── Data valid untuk input: {len(valid_data)}")
    print(f"└── Data tidak valid: {len(df_setortunai_with_extract) - len(valid_data)}")
    
    if len(valid_data) > 0:
        # Tampilkan perbaikan mapping kolom
        print(f"\n📅 === MAPPING KOLOM YANG DIPERBAIKI ===")
        print("Per Bulan (jarak 10 kolom):")
        month_mappings = [
            ("Januari", "P, Q", "M"),
            ("Februari", "Z, AA", "W"),
            ("Maret", "AJ, AK", "AG"),
            ("April", "AT, AU", "AQ"),
            ("Mei", "BD, BE", "BA"),
            ("Juni", "BN, BO", "BK"),
            ("Juli", "BX, BY", "BU"),
            ("Agustus", "CH, CI", "CE"),
            ("September", "CR, CS", "CO"),
            ("Oktober", "DB, DC", "CY"),
            ("November", "DL, DM", "DI"),
            ("Desember", "DV, DW", "DS")
        ]
        
        for bulan, posting_cols, denda_col in month_mappings:
            print(f"├── {bulan:10s}: Posting ({posting_cols:7s}) | Denda ({denda_col})")
        
        # Tampilkan distribusi data
        print(f"\n📈 DISTRIBUSI DATA:")
        print("Per Tahun:")
        year_dist = valid_data['Tahun'].value_counts()
        for year, count in year_dist.items():
            print(f"  - {year}: {count} data")
        
        print("\nPer Sheet (2 digit pertama):")
        valid_data['Sheet_Code'] = valid_data['Kode_8_Digit'].str[:2]
        sheet_dist = valid_data['Sheet_Code'].value_counts()
        for code, count in sheet_dist.items():
            sheet_name = get_sheet_name(code)
            print(f"  - {code} ({sheet_name}): {count} data")
        
        # Sample data yang akan diinput
        print(f"\n🌟 SAMPLE DATA YANG AKAN DIINPUT:")
        sample_cols = ['Kode_8_Digit', 'Bulan', 'Tahun', 'Posting Date', 'Denda']  # Menggunakan kolom Denda
        display(valid_data[sample_cols].head(5))
        
        # Konfirmasi dan jalankan input
        print(f"\n🔄 MEMULAI PROSES INPUT (DENGAN BACKUP & CEK DATA TERISI)...")
        results = input_data_to_excel_v2(valid_data)
        
        # Tampilkan hasil
        print(f"\n🎯 === HASIL INPUT DATA (V2) ===")
        print(f"✅ Berhasil: {results['success']} data")
        print(f"⏭️ Di-skip (sudah terisi): {results['skipped']} data")
        print(f"❌ Gagal: {results['failed']} data")
        
        # Tampilkan detail data yang di-skip
        if results['skipped'] > 0:
            print(f"\n📋 DETAIL DATA YANG DI-SKIP:")
            skip_df = pd.DataFrame(results['skipped_details'])
            display(skip_df.head(10))
        
        if results['errors']:
            print(f"\n🚨 ERROR YANG TERJADI:")
            for i, error in enumerate(results['errors'][:5], 1):  # Tampilkan max 5 error
                print(f"  {i}. {error}")
            if len(results['errors']) > 5:
                print(f"  ... dan {len(results['errors']) - 5} error lainnya")
        
        if results['success'] > 0:
            print(f"\n🎉 INPUT DATA SELESAI!")
            print(f"📁 File Excel backup telah dibuat dan diupdate dengan {results['success']} data baru")
            print(f"🔒 File asli tetap aman dan tidak berubah")
            print(f"🎨 Sel yang berhasil di-input telah diberi warna #D8E4BC")
        
    else:
        print("❌ Tidak ada data valid untuk diinput")
        
else:
    print("❌ Data df_setortunai_with_extract tidak tersedia atau kosong")

📥 === INPUT DATA KE EXCEL MASTER (SAVE AS + CEK DATA TERISI + PEWARNAAN) ===
🚀 === MEMULAI INPUT DATA KE EXCEL MASTER (V2) ===
📊 DATA YANG AKAN DIINPUT:
├── Total data SETORTUNAI dengan extract: 93
├── Data valid untuk input: 93
└── Data tidak valid: 0

📅 === MAPPING KOLOM YANG DIPERBAIKI ===
Per Bulan (jarak 10 kolom):
├── Januari   : Posting (P, Q   ) | Denda (M)
├── Februari  : Posting (Z, AA  ) | Denda (W)
├── Maret     : Posting (AJ, AK ) | Denda (AG)
├── April     : Posting (AT, AU ) | Denda (AQ)
├── Mei       : Posting (BD, BE ) | Denda (BA)
├── Juni      : Posting (BN, BO ) | Denda (BK)
├── Juli      : Posting (BX, BY ) | Denda (BU)
├── Agustus   : Posting (CH, CI ) | Denda (CE)
├── September : Posting (CR, CS ) | Denda (CO)
├── Oktober   : Posting (DB, DC ) | Denda (CY)
├── November  : Posting (DL, DM ) | Denda (DI)
├── Desember  : Posting (DV, DW ) | Denda (DS)

📈 DISTRIBUSI DATA:
Per Tahun:
  - 2025: 89 data
  - 2024: 4 data

Per Sheet (2 digit pertama):
  - 02 (MELONG): 44 

,Kode_8_Digit,Bulan,Tahun,Posting Date,Denda
0,03020308,Juni,2025,01/07/2025,0.0
1,01010205,Juni,2025,01/07/2025,7900.0
2,02040203,Mei,2025,01/07/2025,8950.0
4,02040104,Desember,2024,01/07/2025,0.0
5,02040104,Januari,2025,01/07/2025,0.0



🔄 MEMULAI PROSES INPUT (DENGAN BACKUP & CEK DATA TERISI)...

🔍 === ANALISIS DATA UNTUK INPUT (V2) ===

📅 MEMPROSES TAHUN: 2024
   Data: 4 baris
   💾 Membuat backup file...
   📋 Backup disimpan: Master Data/Master INPUT 2024 FIX_BACKUP_20250721_182204.xlsx
   📂 Membuka file backup untuk diedit...
   ⏭️ Skip 02040104: Data sudah terisi di MELONG row 303
   ⏭️ Skip 02040316: Data sudah terisi di MELONG row 349
   ⏭️ Skip 02030405: Data sudah terisi di MELONG row 256
   ⏭️ Skip 03030211: Data sudah terisi di LG  row 214
   ⏭️ Skip 02040104: Data sudah terisi di MELONG row 303
   ⏭️ Skip 02040316: Data sudah terisi di MELONG row 349
   ⏭️ Skip 02030405: Data sudah terisi di MELONG row 256
   ⏭️ Skip 03030211: Data sudah terisi di LG  row 214
   💾 File backup dengan data baru disimpan: Master Data/Master INPUT 2024 FIX_BACKUP_20250721_182204.xlsx

📅 MEMPROSES TAHUN: 2025
   Data: 89 baris
   💾 Membuat backup file...
   📋 Backup disimpan: Master Data/Master INPUT 2025_BACKUP_20250721_182206.

,Kode_8_Digit,Bulan,Sheet,Row,Reason,Cell1_Filled,Cell2_Filled,Cell3_Filled
0,02040104,Desember,MELONG,303,Data sudah terisi di MELONG row 303,True,True,False
1,02040316,September,MELONG,349,Data sudah terisi di MELONG row 349,True,True,True
2,02030405,November,MELONG,256,Data sudah terisi di MELONG row 256,True,True,True
3,03030211,November,LG,214,Data sudah terisi di LG row 214,True,True,False
4,01010205,Juni,CIGUGUR,19,Data sudah terisi di CIGUGUR row 19,False,False,True
5,02040203,Mei,MELONG,316,Data sudah terisi di MELONG row 316,True,True,True
6,02040104,Januari,MELONG,303,Data sudah terisi di MELONG row 303,True,True,False
7,02040104,Februari,MELONG,303,Data sudah terisi di MELONG row 303,True,True,False
8,02040104,Juni,MELONG,303,Data sudah terisi di MELONG row 303,True,True,False
9,03030518,Juni,LG,293,Data sudah terisi di LG row 293,True,True,False



🎉 INPUT DATA SELESAI!
📁 File Excel backup telah dibuat dan diupdate dengan 2 data baru
🔒 File asli tetap aman dan tidak berubah
🎨 Sel yang berhasil di-input telah diberi warna #D8E4BC


In [10]:
# === EXPORT DATA DENGAN STATUS KE FILE EXCEL TERPISAH ===
print("📤 === EXPORT DATA DENGAN STATUS INPUT ===")

from datetime import datetime
import pandas as pd

# Pastikan variabel results dan valid_data tersedia dari cell sebelumnya
if 'results' in locals() and 'valid_data' in locals() and len(valid_data) > 0:
    
    # Buat dataframe untuk export dengan kolom status
    df_export_status = valid_data.copy()
    df_export_status['Status_Input'] = 'Belum Diproses'
    df_export_status['Keterangan_Input'] = ''
    df_export_status['Nilai_Denda_Input'] = ''
    
    # Update status berdasarkan hasil
    for success_detail in results.get('success_details', []):
        kode = success_detail['Kode_8_Digit']
        bulan = success_detail['Bulan']
        mask = (df_export_status['Kode_8_Digit'] == kode) & (df_export_status['Bulan'] == bulan)
        df_export_status.loc[mask, 'Status_Input'] = 'Berhasil Input'
        df_export_status.loc[mask, 'Keterangan_Input'] = f"Data berhasil diinput ke {success_detail['Sheet']} row {success_detail['Row']}"
        df_export_status.loc[mask, 'Nilai_Denda_Input'] = success_detail['Denda']
    
    for skip_detail in results.get('skipped_details', []):
        kode = skip_detail['Kode_8_Digit']
        bulan = skip_detail['Bulan']
        mask = (df_export_status['Kode_8_Digit'] == kode) & (df_export_status['Bulan'] == bulan)
        df_export_status.loc[mask, 'Status_Input'] = 'Skip - Sudah Terisi'
        df_export_status.loc[mask, 'Keterangan_Input'] = skip_detail['Reason']
    
    for failed_detail in results.get('failed_details', []):
        kode = failed_detail['Kode_8_Digit']
        bulan = failed_detail['Bulan']
        mask = (df_export_status['Kode_8_Digit'] == kode) & (df_export_status['Bulan'] == bulan)
        df_export_status.loc[mask, 'Status_Input'] = 'Gagal'
        df_export_status.loc[mask, 'Keterangan_Input'] = failed_detail['Reason']
    
    # Export ke file Excel
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_filename = f"Status_Input_Excel_{timestamp}.xlsx"
    
    with pd.ExcelWriter(export_filename, engine='openpyxl') as writer:
        df_export_status.to_excel(writer, sheet_name='Status_Input', index=False)
    
    print(f"✅ Data status berhasil diexport ke: {export_filename}")
    print(f"   - Total data: {len(df_export_status)} baris")
    
    # Tampilkan ringkasan status
    status_counts = df_export_status['Status_Input'].value_counts()
    print(f"\n📊 RINGKASAN STATUS INPUT:")
    for status, count in status_counts.items():
        print(f"   ├── {status}: {count} data")
    
    # Export hasil dengan status tracking
    print(f"\n📝 KOLOM KETERANGAN YANG DITAMBAHKAN:")
    print(f"   ├── Status_Input: Status apakah berhasil, skip, atau gagal")
    print(f"   ├── Keterangan_Input: Detail informasi tentang proses input")
    print(f"   └── Nilai_Denda_Input: Nilai denda yang berhasil diinput")
    
    print(f"\n🎉 EXPORT DATA DENGAN STATUS TRACKING SELESAI!")
    
else:
    print("❌ Data tidak tersedia untuk diexport. Pastikan cell sebelumnya sudah dijalankan dengan sukses.")

📤 === EXPORT DATA DENGAN STATUS INPUT ===
✅ Data status berhasil diexport ke: Status_Input_Excel_20250721_175440.xlsx
   - Total data: 93 baris

📊 RINGKASAN STATUS INPUT:
   ├── Skip - Sudah Terisi: 91 data
   ├── Berhasil Input: 2 data

📝 KOLOM KETERANGAN YANG DITAMBAHKAN:
   ├── Status_Input: Status apakah berhasil, skip, atau gagal
   ├── Keterangan_Input: Detail informasi tentang proses input
   └── Nilai_Denda_Input: Nilai denda yang berhasil diinput

🎉 EXPORT DATA DENGAN STATUS TRACKING SELESAI!
